## Introdução ao AgentCore, Strands Agents e A2A

O [protocolo A2A](https://a2a-protocol.org/dev/specification/) é um padrão aberto projetado para facilitar a comunicação e interoperabilidade entre sistemas de agentes de IA independentes e potencialmente opacos. Em um ecossistema onde agentes podem ser construídos usando diferentes frameworks, linguagens ou por diferentes fornecedores, o A2A fornece uma linguagem comum e modelo de interação.

O [Amazon AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) fornece um ambiente de hospedagem seguro, serverless e especialmente desenvolvido para implantar e executar agentes de IA ou ferramentas.

Recentemente, a AWS anunciou [suporte a A2A](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html) para o AgentCore Runtime.

Neste workshop, você construirá a seguinte arquitetura, usando o AgentCore Runtime:

<img src="images/architecture-getting-started.png" style="width: 80%;">

Neste notebook introdutório, vamos construir dois agentes. O primeiro agente é um especialista em documentação AWS. Ele consultará o AWS Docs MCP para ler e pesquisar documentação AWS e também gerar recomendações. O segundo agente é um especialista em blogs AWS. Ele usará pesquisa na web para consultar os últimos blogs e notícias da AWS.

Então vamos começar!

### Configuração

Instalar dependências

In [ ]:
%pip install -q -r requirements.txt --no-cache-dir --force-reinstall

**Por favor, reinicie seu ambiente para que ele possa refletir as novas versões!**

In [ ]:
#import IPython

#IPython.Application.instance().kernel.do_shutdown(True)

Verificando se a versão do `bedrock-agentcore-starter-toolkit` é 0.1.21

In [ ]:
!pip freeze | grep boto
!pip freeze | grep agentcore

In [ ]:
# Import libraries
import os
import json
import requests
import boto3
import time
from boto3.session import Session
from strands.tools import tool

# Get boto session
boto_session = Session()

### 1 - Criar código para os dois agentes

Criar pasta `agents` se ela não estiver criada.

In [ ]:
![ ! -d "agents" ] && mkdir agents

#### 1.1 - Agente especialista em documentação AWS

Primeiro, vamos escrever o código do nosso primeiro agente em um arquivo local; este agente será posteriormente implantado no AgentCore runtime.

In [ ]:
%%writefile agents/strands_aws_docs.py
import os
import logging
import asyncio
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.multiagent.a2a import A2AServer
from strands.tools.mcp import MCPClient
from fastapi import FastAPI
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000

# Cliente MCP global com inicialização preguiçosa
_mcp_client = None

async def get_mcp_client():
    """Inicialização preguiçosa do cliente MCP com timeout"""
    global _mcp_client
    if _mcp_client is None:
        try:
            _mcp_client = MCPClient(
                lambda: stdio_client(
                    StdioServerParameters(
                        command="uvx", 
                        args=["awslabs.aws-documentation-mcp-server@latest"]
                    )
                )
            )
            # Iniciar com timeout
            await asyncio.wait_for(_mcp_client.start(), timeout=10.0)
            logger.info("Cliente MCP inicializado")
        except asyncio.TimeoutError:
            logger.error("Timeout na inicialização do cliente MCP")
            _mcp_client = None
        except Exception as e:
            logger.error(f"Falha no cliente MCP: {e}")
            _mcp_client = None
    return _mcp_client

system_prompt = """Você é um Assistente de Documentação AWS alimentado pelo servidor MCP de Documentação AWS. Seu papel é ajudar usuários a encontrar informações precisas e atualizadas da documentação AWS.

CRÍTICO: Mantenha respostas CURTAS e FOCADAS.

Diretrizes:
- Forneça respostas concisas e práticas (máximo 3 frases)
- Use marcadores para listas
- Pule explicações verbosas
- Se o MCP estiver indisponível, forneça conhecimento básico da AWS
- Timeout de operações após 8 segundos
- Priorize velocidade sobre completude

Você tem acesso às ferramentas de pesquisa de documentação AWS quando disponíveis."""

# Inicializar agente com ferramentas mínimas primeiro
agent = Agent(
    system_prompt=system_prompt, 
    tools=[],  # Iniciar sem ferramentas, adicionar dinamicamente
    name="AWS Docs Agent",
    description="Um agente para consultar Docs AWS usando AWS MCP.",
)

# Adicionar ferramentas dinamicamente quando o MCP estiver pronto
async def setup_agent_tools():
    """Configurar ferramentas do agente quando o cliente MCP estiver pronto"""
    try:
        mcp_client = await get_mcp_client()
        if mcp_client:
            tools = await asyncio.wait_for(
                mcp_client.list_tools_async(), 
                timeout=5.0
            )
            agent.tools = [tools] if tools else []
            logger.info("Ferramentas do agente configuradas")
    except Exception as e:
        logger.warning(f"Não foi possível configurar ferramentas MCP: {e}")

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

@app.on_event("startup")
async def startup_event():
    """Inicializar cliente MCP na inicialização"""
    await setup_agent_tools()

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

#### **Opcional** - Teste local

Se você quiser testar este código localmente, pode abrir uma janela bash/terminal e executar os seguintes comandos:

```bash
python agents/strands_aws_docs.py
```

O servidor iniciará localmente. Em seguida, execute em outro terminal/bash o seguinte comando para testá-lo:

```bash
curl -X POST http://0.0.0.0:9000 \-H "Content-Type: application/json" \-d '{  "jsonrpc": "2.0",  "id": "req-001",  "method": "message/send",  "params": {  "message": {  "role": "user",  "parts": [  {  "kind": "text",  "text": "O que é AWS Lambda?"  }  ],  "messageId": "d0673ab9-796d-4270-9435-451912020cd1"  }  } }' | jq .
```

Ele consultará o MCP e retornará uma resposta explicando o AWS Lambda.

Você também pode testar a recuperação de informações do agent card, usando o seguinte comando:

```bash
curl http://localhost:9000/.well-known/agent-card.json | jq .
```

#### 1.2 - Agente especialista em blogs AWS

Agora, vamos escrever o código do nosso segundo agente em um arquivo local.

In [ ]:
%%writefile agents/strands_aws_blogs_news.py
import logging
import os
import asyncio
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
import uvicorn
from fastapi import FastAPI

from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')

@tool
async def fast_internet_search(keywords: str, max_results: int = 3) -> str:
    """Pesquisa rápida na web com timeouts.
    Args:
        keywords (str): Palavras-chave da consulta de pesquisa
        max_results (int): Máximo de resultados (padrão 3 para velocidade)
    Returns:
        Resultados da pesquisa
    """
    try:
        # Adicionar termos específicos da AWS para melhores resultados
        aws_keywords = f"site:aws.amazon.com {keywords} AWS"
        
        # Usar timeout do asyncio para a pesquisa
        async def search_with_timeout():
            return DDGS().text(
                aws_keywords, 
                region="us-en", 
                max_results=max_results
            )
        
        results = await asyncio.wait_for(search_with_timeout(), timeout=8.0)
        
        if results:
            # Formatar resultados de forma concisa
            formatted = []
            for i, result in enumerate(results[:max_results], 1):
                formatted.append(f"{i}. {result.get('title', 'Sem título')}\n   {result.get('href', '')}")
            
            return "\n".join(formatted)
        else:
            return "Nenhum resultado AWS encontrado."
            
    except asyncio.TimeoutError:
        logger.warning(f"Timeout na pesquisa para: {keywords}")
        return "Pesquisa expirou. Tente uma consulta mais específica."
    except RatelimitException:
        logger.warning("Limite de taxa atingido")
        return "Limite de taxa atingido. Por favor, tente novamente em instantes."
    except (DDGSException, Exception) as e:
        logger.error(f"Erro na pesquisa: {e}")
        return f"Pesquisa indisponível: {str(e)[:50]}"

system_prompt = """Você é um Especialista em Blogs AWS.

CRÍTICO: Mantenha respostas CURTAS e RECENTES.

Diretrizes:
- Forneça no máximo 3 resultados recentes
- Foque apenas em posts oficiais do blog AWS
- Use resumos concisos (1-2 frases por resultado)
- Inclua links diretos quando disponíveis
- Timeout de pesquisas após 8 segundos
- Se a pesquisa falhar, reconheça a limitação

Estratégia de Pesquisa:
- Sempre inclua "AWS" nas pesquisas
- Foque em conteúdo de aws.amazon.com/blogs/
- Priorize anúncios recentes"""

agent = Agent(
    system_prompt=system_prompt, 
    tools=[fast_internet_search],
    name="AWS Blog/News Agent",
    description="Um agente para pesquisar na web os últimos blogs e notícias AWS.",
)

host, port = "0.0.0.0", 9000

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

Vamos criar um arquivo requirements.txt com as dependências necessárias para o agente.

In [ ]:
%%writefile agents/requirements.txt
boto3==1.40.50
bedrock-agentcore==0.1.7
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs

### 2 - Implantar no AgentCore Runtime

Agora, vamos implantar esta solução no AgentCore Runtime.

#### 2.1 - Configurar Cognito User Pool

Antes de implantar os agentes, precisamos configurar um Cognito User Pool, para que ele possa validar os usuários que estão acessando nossos agentes, ou qualquer outro provedor de identidade como Okta, Microsoft Entra ID, etc.

Vamos importar uma classe auxiliar, que tem métodos para simplificar alguns passos em nosso workshop. Esta classe auxiliar importará métodos responsáveis por criar o Cognito User Pool

In [ ]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = (
    setup_cognito_user_pool()
)  # You'll get your bearer token from this output cell.
print("Cognito setup completed ✓")

#### 2.2 - Criar função IAM para os agentes

##### 2.2.1 Função de execução do agente AWS Docs

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_DOCS_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)

##### 2.2.2 Função de execução do agente AWS Blogs

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_BLOG_ROLE_NAME

execution_role_arn_blogs = create_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)

##### Criar configurações para implantação no AgentCore Runtime

Na seção seguinte, estamos aproveitando o [starter toolkit](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-starter-toolkit.html). O starter toolkit é uma ferramenta de interface de linha de comando (CLI) que você pode usar para implantar agentes de IA em um AgentCore Runtime.

Agora criaremos um agente com suporte para protocolo A2A dentro do AgentCore runtime.

##### 2.2.3 - Vamos configurar e implantar nosso primeiro agente (Agente AWS Docs):

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_mcp_agent = Runtime()
aws_docs_agent_name="aws_docs_assistant"

region = boto_session.region_name

# Configure the deployment
response_aws_docs_agent = agentcore_runtime_mcp_agent.configure(
    entrypoint="agents/strands_aws_docs.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_docs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("Configuration completed:", response_aws_docs_agent)

Iniciar o primeiro agente no AgentCore Runtime

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

docs_agent_arn = launch_result_mcp.agent_arn

**Verificar status da implantação**

Vamos verificar se a implantação foi concluída:

In [ ]:
status_response = agentcore_runtime_mcp_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

##### 2.2.4 - Vamos configurar e implantar nosso segundo agente (Agente de blogs e notícias AWS):

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_blogs = Runtime()
aws_blogs_agent_name="aws_blog_assistant"

# Configure the deployment
response_aws_blogs_agent = agentcore_runtime_blogs.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn_blogs,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_blogs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A"
)

print("Configuration completed:", response_aws_blogs_agent)

Iniciar o segundo agente no AgentCore Runtime

In [ ]:
launch_result_blog = agentcore_runtime_blogs.launch()
print("Launch completed:", launch_result_blog.agent_arn)

blog_agent_arn = launch_result_blog.agent_arn

**Verificar status da implantação**

Vamos verificar se a implantação do segundo agente foi concluída:

In [ ]:
status_response = agentcore_runtime_blogs.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

##### 2.2.5 - Exportar e salvar saídas

Exportar variáveis para serem usadas nos próximos notebooks:

In [ ]:
MCP_AGENT_ID = launch_result_mcp.agent_id
MCP_AGENT_ARN = launch_result_mcp.agent_arn
MCP_AGENT_NAME = aws_docs_agent_name

BLOG_AGENT_ID = launch_result_blog.agent_id
BLOG_AGENT_ARN = launch_result_blog.agent_arn
BLOG_AGENT_NAME = aws_blogs_agent_name

COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store MCP_AGENT_ID
%store MCP_AGENT_ARN
%store MCP_AGENT_NAME
%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

Armazenar ARN dos agentes no SSM, para que possa ser usado pelo orquestrador:

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, MCP_AGENT_ARN)

put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

### 3 - Invocar agentes A2A

Primeiro, vamos atualizar o token de autenticação:

In [ ]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

#### 3.1 Obtendo Agent Cards

Agora vamos começar obtendo as informações do Agent Card do nosso primeiro agente (Especialista em AWS Docs MCP):

In [ ]:
import logging
from uuid import uuid4
from urllib.parse import quote

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

def fetch_agent_card(agent_arn):
    # Codificar URL do ARN do agente
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construir a URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    logger.info(url)
    # Gerar um ID de sessão único
    session_id = str(uuid4())
    logger.info(f"ID de sessão gerado: {session_id}")

    # Definir cabeçalhos
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # Fazer a solicitação
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Analisar e imprimir JSON formatado
        agent_card = response.json()
        logger.info(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        logger.error(f"Erro ao buscar agent card: {e}")
        return None

In [ ]:
fetch_agent_card(docs_agent_arn)

Agora vamos verificar o agent card para o segundo agente (especialista em blogs e notícias AWS):

In [ ]:
fetch_agent_card(blog_agent_arn)

#### 3.2 - Testar agentes

Agora, vamos invocar o primeiro agente, usando A2A:

In [ ]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # definir timeout de requisição para 5 minutos

def format_agent_response(response):
    """Extrair e formatar resposta do agente para legibilidade humana."""
    # Obter o texto de resposta principal dos artefatos
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text
    
    # Fallback: concatenar todas as mensagens do agente do histórico
    agent_messages = [
        msg.parts[0].root.text 
        for msg in response.history 
        if msg.role.value == 'agent' and msg.parts
    ]
    return ''.join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(agent_arn, message: str):
    # Obter URL de runtime da variável de ambiente
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construir a URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # Gerar um ID de sessão único
    session_id = str(uuid4())
    print(f"ID de sessão gerado: {session_id}")

    # Adicionar cabeçalhos de autenticação para AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Obter agent card da URL de runtime
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contém a URL correta (mesma que runtime_url neste caso)
        # Nenhuma substituição manual necessária - este é o padrão de montagem baseado em caminho

        # Criar cliente usando factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Usar modo não-streaming para resposta síncrona
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Criar e enviar mensagem
        msg = create_message(text=message)

        # Com streaming=False, isso retornará exatamente um resultado
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # tupla (Task, UpdateEvent)
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback para outros tipos de resposta
                logger.info(f"Resposta: {str(event)}")
                return event

In [ ]:
result = await send_sync_message(docs_agent_arn, "what is DynamoDB")
formatted_output = format_agent_response(result)
print(formatted_output)

Agora, vamos testar nosso segundo agente:

In [ ]:
result = await send_sync_message(blog_agent_arn, "Give me the latest published blog for Bedrock AgentCore?")
formatted_output = format_agent_response(result)
print(formatted_output)

A seguir está uma saída mais detalhada, mostrando os passos que o agente executou.

Sinta-se à vontade para mudar as perguntas feitas ao agente e ver o resultado passo a passo.

In [ ]:
def format_agent_trace(response):
    """Formatar resposta do agente como um rastreamento legível de chamadas."""
    print("=" * 60)
    print("🔍 RASTREAMENTO DE EXECUÇÃO DO AGENTE")
    print("=" * 60)
    
    # Informações de contexto
    print(f"📋 ID de contexto: {response.context_id}")
    print(f"🆔 ID de tarefa: {response.id}")
    print(f"📊 Status: {response.status.state.value}")
    print(f"⏰ Concluído: {response.status.timestamp}")
    print()
    
    # Rastrear através do histórico
    print("🔄 FLUXO DE EXECUÇÃO:")
    print("-" * 40)
    
    for i, msg in enumerate(response.history, 1):
        role_icon = "👤" if msg.role.value == "user" else "🤖"
        text = msg.parts[0].root.text if msg.parts else "[Sem conteúdo]"
        
        # Truncar mensagens longas para visualização de rastreamento
        if len(text) > 80:
            text = text[:77] + "..."
            
        print(f"{i:2d}. {role_icon} {msg.role.value.upper()}: {text}")
    
    print()
    print("✅ RESULTADO FINAL:")
    print("-" * 40)
    
    # Artefato final
    if response.artifacts:
        final_text = response.artifacts[0].parts[0].root.text
        print(final_text[:200] + "..." if len(final_text) > 200 else final_text)
    
    print("=" * 60)

In [ ]:
format_agent_trace(result)

Parabéns, você implantou seu primeiro agente, usando protocolo A2A no Amazon AgentCore Runtime!

Agora, vamos para o próximo laboratório.